## Ingesting Data

**1\. Download the datasets from source [here](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce/data?select=olist\_customers\_dataset.csv).**

*   Or you can download to your local machine using curl command:<br>
    * _Create a directory for the file:_ <br>
    <code>mkdir olist</code>
    * _Download the file to the created directory:_ <br>
    <code>curl -L -o ~/olist/brazilian-ecommerce.zip\
        https://www.kaggle.com/api/v1/datasets/download/olistbr/brazilian-ecommerce
    </code>
    * _Create a data directory in the olist folder:_<br>
    <code>mkdir ~/olist/data
    </code>
    * _Unzip the dowloaded file into the data folder:_<br>
    <code>unzip ~/olist/brazilian-ecommerce.zip -d ~/olist/data/
    </code>
    
**2\. Ingest Data in HDFS and not local.**

*   Steps:<br>
    * _Create a directory on hdfs:_<br>
    <code>hdfs dfs -mkdir /user/Marho/olist
    </code>
    * _Move olist data from local to hdfs_:<br>
    <code> hdfs dfs -put ~/olist/data/ /user/Marho/olist
    </code>
    * _Confirm if files has been loaded successfully_:<br>
    <code> hdfs dfs -ls /user/Marho/olist/data
    </code>



## Data Exploration with spark Cluster (GCP Dataproc)

Creating a spark session for data interaction

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('Olist Data Exploration').getOrCreate()
spark

26/06/24 07:51:00 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


### Load Data to Spark Dataframe

In [2]:
hdfs_base_path = '/user/Marho/project/olist/data/'

In [3]:
options = {
    'header': True,
    'inferSchema': True
}

customer_df = spark.read.format('csv').options(**options).load(hdfs_base_path+'olist_customers_dataset.csv')
geolocation_df = spark.read.format('csv').options(**options).load(hdfs_base_path+ 'olist_geolocation_dataset.csv')
order_items_df = spark.read.format('csv').options(**options).load(hdfs_base_path+'olist_order_items_dataset.csv')
payment_df = spark.read.format('csv').options(**options).load(hdfs_base_path+'olist_order_payments_dataset.csv')
review_df = spark.read.format('csv').options(**options).load(hdfs_base_path+'olist_order_reviews_dataset.csv')
orders_df = spark.read.format('csv').options(**options).load(hdfs_base_path+'olist_orders_dataset.csv')
products_df = spark.read.format('csv').options(**options).load(hdfs_base_path+'olist_products_dataset.csv')
sellers_df = spark.read.format('csv').options(**options).load(hdfs_base_path+'olist_sellers_dataset.csv')
products_category_df = spark.read.format('csv').options(**options).load(hdfs_base_path+'product_category_name_translation.csv')

### Checking for Data Leakage with Source as at 23/06/2026
- customers dataset comprises of : 99441 values
- geolocation dataset comprises of: 1000163 values
- order items dataset comprises of: 112650 values
- order payments dataset comprises of: 103886 values
- order reviews dataset comprises of: 104162
- orders dataset comprises of: 99441 values
- products dataset comprises of: 32951 values
- sellers dataset comprises of: 3095 values
- product category dataset of: 71 values

In [10]:
print(f'Total rows in customers dataset: {customer_df.count()}')
print(f'Total rows in geolocation dataset: {geolocation_df.count()}')
print(f'Total rows in order items dataset: {order_items_df.count()}')
print(f'Total rows in order payments dataset: {payment_df.count()}')
print(f'Total rows in order reviews dataset: {review_df.count()}')
print(f'Total rows in orders dataset: {orders_df.count()}')
print(f'Total rows in products dataset: {products_df.count()}')
print(f'Total rows in sellers dataset: {sellers_df.count()}')
print(f'Total rows in product category name translation dataset: {products_category_df.count()}')

Total rows in customers dataset: 99441


Total rows in geolocation dataset: 1000163


Total rows in order items dataset: 112650


Total rows in order payments dataset: 103886


Total rows in order reviews dataset: 104162


Total rows in orders dataset: 99441


Total rows in products dataset: 32951


Total rows in sellers dataset: 3095


Total rows in product category name translation dataset: 71


### Checking for Null

In [8]:
from pyspark.sql.functions import  *

customer_df.select([count(when(col(c).isNull(),1)).alias(c) for c in customer_df.columns]).show()
geolocation_df.select([count(when(col(c).isNull(),1)).alias(c) for c in geolocation_df.columns]).show()
order_items_df.select([count(when(col(c).isNull(),1)).alias(c) for c in order_items_df.columns]).show()
payment_df.select([count(when(col(c).isNull(),1)).alias(c) for c in payment_df.columns]).show()
review_df.select([count(when(col(c).isNull(),1)).alias(c) for c in review_df.columns]).show()
orders_df.select([count(when(col(c).isNull(),1)).alias(c) for c in orders_df.columns]).show()
products_df.select([count(when(col(c).isNull(),1)).alias(c) for c in products_df.columns]).show()
sellers_df.select([count(when(col(c).isNull(),1)).alias(c) for c in sellers_df.columns]).show()
products_category_df.select([count(when(col(c).isNull(),1)).alias(c) for c in products_category_df.columns]).show()

+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|          0|                 0|                       0|            0|             0|
+-----------+------------------+------------------------+-------------+--------------+



+---------------------------+---------------+---------------+----------------+-----------------+
|geolocation_zip_code_prefix|geolocation_lat|geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+---------------+---------------+----------------+-----------------+
|                          0|              0|              0|               0|                0|
+---------------------------+---------------+---------------+----------------+-----------------+



+--------+-------------+----------+---------+-------------------+-----+-------------+
|order_id|order_item_id|product_id|seller_id|shipping_limit_date|price|freight_value|
+--------+-------------+----------+---------+-------------------+-----+-------------+
|       0|            0|         0|        0|                  0|    0|            0|
+--------+-------------+----------+---------+-------------------+-----+-------------+



+--------+------------------+------------+--------------------+-------------+
|order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------+------------------+------------+--------------------+-------------+
|       0|                 0|           0|                   0|            0|
+--------+------------------+------------+--------------------+-------------+



+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|        1|    2236|        2380|               92157|                 63079|                8764|                   8785|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+



+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|         0|                  610|                610|                       610|               610|               2|                2|                2|               2|
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+



+---------+----------------------+-----------+------------+
|seller_id|seller_zip_code_prefix|seller_city|seller_state|
+---------+----------------------+-----------+------------+
|        0|                     0|          0|           0|
+---------+----------------------+-----------+------------+



+---------------------+-----------------------------+
|product_category_name|product_category_name_english|
+---------------------+-----------------------------+
|                    0|                            0|
+---------------------+-----------------------------+



### Checking for Duplicate

In [12]:
customer_df.groupBy(customer_df.columns).count().filter(col("count") > 1).show(truncate=False)
geolocation_df.groupBy(geolocation_df.columns).count().filter(col("count") > 1).show(truncate=False)
order_items_df.groupBy(order_items_df.columns).count().filter(col("count") > 1).show(truncate=False)
payment_df.groupBy(payment_df.columns).count().filter(col("count") > 1).show(truncate=False)
review_df.groupBy(review_df.columns).count().filter(col("count") > 1).show(truncate=False)
orders_df.groupBy(orders_df.columns).count().filter(col("count") > 1).show(truncate=False)
products_df.groupBy(products_df.columns).count().filter(col("count") > 1).show(truncate=False)
sellers_df.groupBy(sellers_df.columns).count().filter(col("count") > 1).show(truncate=False)
products_category_df.groupBy(products_category_df.columns).count().filter(col("count") > 1).show(truncate=False)

+-----------+------------------+------------------------+-------------+--------------+-----+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|count|
+-----------+------------------+------------------------+-------------+--------------+-----+
+-----------+------------------+------------------------+-------------+--------------+-----+



+---------------------------+-------------------+-------------------+----------------+-----------------+-----+
|geolocation_zip_code_prefix|geolocation_lat    |geolocation_lng    |geolocation_city|geolocation_state|count|
+---------------------------+-------------------+-------------------+----------------+-----------------+-----+
|1106                       |-23.531062064948905|-46.629505278925556|sao paulo       |SP               |2    |
|1241                       |-23.545792634196992|-46.6567234730798  |sao paulo       |SP               |3    |
|1532                       |-23.57418642936694 |-46.6357481924808  |são paulo       |SP               |3    |
|2180                       |-23.51796174271452 |-46.56586340175424 |sao paulo       |SP               |3    |
|2168                       |-23.52218624279628 |-46.57970414962962 |sao paulo       |SP               |2    |
|2519                       |-23.50699116126142 |-46.667994993479134|sao paulo       |SP               |2    |
|

+--------+-------------+----------+---------+-------------------+-----+-------------+-----+
|order_id|order_item_id|product_id|seller_id|shipping_limit_date|price|freight_value|count|
+--------+-------------+----------+---------+-------------------+-----+-------------+-----+
+--------+-------------+----------+---------+-------------------+-----+-------------+-----+



+--------+------------------+------------+--------------------+-------------+-----+
|order_id|payment_sequential|payment_type|payment_installments|payment_value|count|
+--------+------------------+------------+--------------------+-------------+-----+
+--------+------------------+------------+--------------------+-------------+-----+



+----------------------------------------------------------------------------------------------+---------------------------------------------------------------+-------------------+--------------------+----------------------+--------------------+-----------------------+-----+
|review_id                                                                                     |order_id                                                       |review_score       |review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|count|
+----------------------------------------------------------------------------------------------+---------------------------------------------------------------+-------------------+--------------------+----------------------+--------------------+-----------------------+-----+
|POREM NAO RECEBEMOS A MANTA                                                                   | SOMENTE OS PROTETOR DE COLUNA"                                |2018-02-22 0

+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+-----+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|count|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+-----+
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+-----+



+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+-----+
|product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|count|
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+-----+
+----------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+-----+



+---------+----------------------+-----------+------------+-----+
|seller_id|seller_zip_code_prefix|seller_city|seller_state|count|
+---------+----------------------+-----------+------------+-----+
+---------+----------------------+-----------+------------+-----+



+---------------------+-----------------------------+-----+
|product_category_name|product_category_name_english|count|
+---------------------+-----------------------------+-----+
+---------------------+-----------------------------+-----+



## Customer Distributuion 

### Customer Distributuion across city

In [19]:
customer_cities_stat = (customer_df.groupBy('customer_city')
 .agg(count('customer_id').alias('count'))
 .orderBy(desc('count'))
 )

customer_cities_stat.show()

+--------------------+-----+
|       customer_city|count|
+--------------------+-----+
|           sao paulo|15540|
|      rio de janeiro| 6882|
|      belo horizonte| 2773|
|            brasilia| 2131|
|            curitiba| 1521|
|            campinas| 1444|
|        porto alegre| 1379|
|            salvador| 1245|
|           guarulhos| 1189|
|sao bernardo do c...|  938|
|             niteroi|  849|
|         santo andre|  797|
|              osasco|  746|
|              santos|  713|
|             goiania|  692|
| sao jose dos campos|  691|
|           fortaleza|  654|
|            sorocaba|  633|
|              recife|  613|
|       florianopolis|  570|
+--------------------+-----+
only showing top 20 rows



### Customer Distributuion across state

In [18]:
customer_states_stat = (customer_df.groupBy('customer_state')
 .agg(count('customer_id').alias('count'))
 .orderBy(desc('count'))
 )

customer_states_stat.show()

+--------------+-----+
|customer_state|count|
+--------------+-----+
|            SP|41746|
|            RJ|12852|
|            MG|11635|
|            RS| 5466|
|            PR| 5045|
|            SC| 3637|
|            BA| 3380|
|            DF| 2140|
|            ES| 2033|
|            GO| 2020|
|            PE| 1652|
|            CE| 1336|
|            PA|  975|
|            MT|  907|
|            MA|  747|
|            MS|  715|
|            PB|  536|
|            PI|  495|
|            RN|  485|
|            AL|  413|
+--------------+-----+
only showing top 20 rows



### Customer Distributuion across city and state

In [17]:
customer_cities_n_states_stat = (customer_df.groupBy('customer_city', 'customer_state')
 .agg(count('customer_id').alias('count'))
 .orderBy(desc('count'))
 .show())

customer_cities_n_states_stat.show()

+--------------------+--------------+-----+
|       customer_city|customer_state|count|
+--------------------+--------------+-----+
|           sao paulo|            SP|15540|
|      rio de janeiro|            RJ| 6882|
|      belo horizonte|            MG| 2773|
|            brasilia|            DF| 2131|
|            curitiba|            PR| 1521|
|            campinas|            SP| 1444|
|        porto alegre|            RS| 1379|
|            salvador|            BA| 1245|
|           guarulhos|            SP| 1189|
|sao bernardo do c...|            SP|  938|
|             niteroi|            RJ|  849|
|         santo andre|            SP|  796|
|              osasco|            SP|  746|
|              santos|            SP|  713|
|             goiania|            GO|  692|
| sao jose dos campos|            SP|  691|
|           fortaleza|            CE|  654|
|            sorocaba|            SP|  633|
|              recife|            PE|  613|
|       florianopolis|          

## Order Distributuion 

### Order Distributuion by status

In [22]:
orders_df.show(5)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [26]:
order_status_stat = (orders_df.groupBy('order_status')
 .agg(count('order_id').alias('count'))
 .orderBy(desc('count'))
)
order_status_stat.show()

+------------+-----+
|order_status|count|
+------------+-----+
|   delivered|96478|
|     shipped| 1107|
|    canceled|  625|
| unavailable|  609|
|    invoiced|  314|
|  processing|  301|
|     created|    5|
|    approved|    2|
+------------+-----+



### Order delivery duration

In [29]:
delivery_duration = orders_df.withColumn('delivery_duration', date_diff(col('order_delivered_customer_date'), col('order_purchase_timestamp'))).orderBy('delivery_duration', ascending=False)
delivery_duration.select(['order_id','order_status', 'order_purchase_timestamp', 'order_delivered_customer_date', 'delivery_duration']).show()                                         

+--------------------+------------+------------------------+-----------------------------+-----------------+
|            order_id|order_status|order_purchase_timestamp|order_delivered_customer_date|delivery_duration|
+--------------------+------------+------------------------+-----------------------------+-----------------+
|ca07593549f1816d2...|   delivered|     2017-02-21 23:31:27|          2017-09-19 14:36:39|              210|
|1b3190b2dfa9d789e...|   delivered|     2018-02-23 14:57:35|          2018-09-19 23:24:07|              208|
|440d0d17af552815d...|   delivered|     2017-03-07 23:59:51|          2017-09-19 15:12:50|              196|
|2fb597c2f772eca01...|   delivered|     2017-03-08 18:09:02|          2017-09-19 14:33:17|              195|
|285ab9426d6982034...|   delivered|     2017-03-08 22:47:40|          2017-09-19 14:00:04|              195|
|0f4519c5f1c541dde...|   delivered|     2017-03-09 13:26:57|          2017-09-19 14:38:21|              194|
|47b40429ed8cce3ae.

In [31]:
delivery_duration.groupby('order_status').agg(count('order_id')).show()

+------------+---------------+
|order_status|count(order_id)|
+------------+---------------+
|   delivered|          96478|
|    canceled|            625|
|     created|              5|
|     shipped|           1107|
|    approved|              2|
|  processing|            301|
|    invoiced|            314|
| unavailable|            609|
+------------+---------------+



## Payment Distribution

In [33]:
payment_type_stat = (payment_df.groupBy('payment_type')
                    .count()
                    .orderBy('count', ascending=False))

payment_type_stat.show()

+------------+-----+
|payment_type|count|
+------------+-----+
| credit_card|76795|
|      boleto|19784|
|     voucher| 5775|
|  debit_card| 1529|
| not_defined|    3|
+------------+-----+



## Order Items Distribution

### Most ordered products

In [15]:
totalProductOrdered = order_items_df.groupBy('product_id').agg(count('order_id').alias('total_order')).orderBy('total_order', ascending=False)
totalProductOrdered.show()

+--------------------+-----------+
|          product_id|total_order|
+--------------------+-----------+
|aca2eb7d00ea1a7b8...|        527|
|99a4788cb24856965...|        488|
|422879e10f4668299...|        484|
|389d119b48cf3043d...|        392|
|368c6c730842d7801...|        388|
|53759a2ecddad2bb8...|        373|
|d1c427060a0f73f6b...|        343|
|53b36df67ebb7c415...|        323|
|154e7e31ebfa09220...|        281|
|3dd2a17168ec895c7...|        274|
|2b4609f8948be1887...|        260|
|7c1bd920dbdf22470...|        231|
|a62e25e09e05e6faf...|        226|
|5a848e4ab52fd5445...|        197|
|bb50f2e236e5eea01...|        195|
|e0d64dcfaa3b6db5c...|        194|
|42a2c92a0979a949c...|        183|
|e53e557d5a159f5aa...|        183|
|b532349fe46b38fbc...|        169|
|35afc973633aaeb6b...|        165|
+--------------------+-----------+
only showing top 20 rows



### Top selling products

In [23]:
order_items_df.show(5)

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30|199.0|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18|12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|2017-02-13 13:57:51|199.9|        18.14|
+--------------------+-------------+------------

In [25]:
topSellingProducts =  (order_items_df.groupBy('product_id')
                       .agg(sum('price').alias('total_amount_spent'))
                       .orderBy('total_amount_spent', ascending=False))                                                            
topSellingProducts.select('product_id', format_number('total_amount_spent',2).alias('total_amount_spent ($)')).show()

+--------------------+----------------------+
|          product_id|total_amount_spent ($)|
+--------------------+----------------------+
|bb50f2e236e5eea01...|             63,885.00|
|6cdd53843498f9289...|             54,730.20|
|d6160fb7873f18409...|             48,899.34|
|d1c427060a0f73f6b...|             47,214.51|
|99a4788cb24856965...|             43,025.56|
|3dd2a17168ec895c7...|             41,082.60|
|25c38557cf793876c...|             38,907.32|
|5f504b3a1c75b73d6...|             37,733.90|
|53b36df67ebb7c415...|             37,683.42|
|aca2eb7d00ea1a7b8...|             37,608.90|
|e0d64dcfaa3b6db5c...|             31,786.82|
|d285360f29ac7fd97...|             31,623.81|
|7a10781637204d8d1...|             30,467.50|
|f1c7f353075ce59d8...|             29,997.36|
|f819f0c84a64f02d3...|             29,024.48|
|588531f8ec37e7d5f...|             28,291.99|
|422879e10f4668299...|             26,577.22|
|16c4e87b98a9370a9...|             25,034.00|
|5a848e4ab52fd5445...|            

In [ ]:
spark.stop()